# Kronos PoC: Kronos-small vs GBM on Japanese Stocks

Spec: `docs/superpowers/specs/2026-05-02-kronos-poc-design.md`. Run all cells top to bottom. See `notebooks/README.md` for details.

## 1. Setup

In [ ]:
# Cell 1: setup
!pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 tqdm==4.67.1
# torch / numpy / pandas / matplotlib are pre-installed on Colab; do not pin to avoid breaking Colab base env.

import os, json, time, hashlib, pickle, getpass
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from tqdm.auto import tqdm

print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}, mps={torch.backends.mps.is_available()}")

## 2. Auth

In [ ]:
# Cell 2: auth
# Try Colab Secrets first, fall back to interactive getpass. The key never gets printed.
try:
    from google.colab import userdata
    JQUANTS_API_KEY = userdata.get("JQUANTS_API_KEY")
    print("Loaded JQUANTS_API_KEY from Colab Secrets.")
except Exception:
    JQUANTS_API_KEY = None

if not JQUANTS_API_KEY:
    JQUANTS_API_KEY = getpass.getpass("JQuants API key (Premium plan): ").strip()

assert JQUANTS_API_KEY, "API key required"
os.environ["JQUANTS_API_KEY"] = JQUANTS_API_KEY
print(f"API key loaded ({len(JQUANTS_API_KEY)} chars). Header will be sent as 'x-api-key: ****'.")

## 3. Fetch OHLCV (~4 years)

In [ ]:
# Cell 3: fetch_data
STOCKS = {"7203": "Toyota", "6758": "Sony Group", "9984": "SoftBank Group"}
JQUANTS_BASE = "https://api.jquants.com/v2"
LOOKBACK = 256
HORIZON = 20
N_WINDOWS = 36
STRIDE = 21
SAMPLE_COUNT = 30
# Calendar-day budget for ~1011 business days (allow 50% headroom for weekends/holidays).
FETCH_DAYS_CAL = int((LOOKBACK + (N_WINDOWS - 1) * STRIDE + HORIZON) * 1.5) + 30

def fetch_ohlcv(code4: str) -> pd.DataFrame:
    """Fetch ~4 years of daily OHLCV from JQuants v2. Returns DataFrame indexed by Date with O/H/L/C/V columns."""
    code5 = code4 + "0"  # 4-digit -> 5-digit
    today = datetime.utcnow().date()
    start = (today - pd.Timedelta(days=FETCH_DAYS_CAL)).isoformat()
    end = today.isoformat()
    url = f"{JQUANTS_BASE}/equities/bars/daily"
    params = {"code": code5, "from": start, "to": end}
    headers = {"x-api-key": os.environ["JQUANTS_API_KEY"]}
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            data = r.json().get("data", [])
            if not data:
                raise RuntimeError(f"No data returned for {code4}")
            df = pd.DataFrame(data)
            df["Date"] = pd.to_datetime(df["Date"])
            df = df.set_index("Date").sort_index()
            df = df.rename(columns={"AdjO": "open", "AdjH": "high", "AdjL": "low", "AdjC": "close", "AdjVo": "volume"})
            return df[["open", "high", "low", "close", "volume"]].astype(float)
        except requests.HTTPError as e:
            if e.response.status_code in (401, 403):
                raise RuntimeError("JQuants auth failed — check API key and Premium plan status") from e
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("unreachable")

df_dict = {}
for code in STOCKS:
    df_dict[code] = fetch_ohlcv(code)
    print(f"{code} {STOCKS[code]}: rows={len(df_dict[code])}, range={df_dict[code].index.min().date()}..{df_dict[code].index.max().date()}")
    time.sleep(0.5)

# Sanity: enough rows for the full window plan
need = LOOKBACK + (N_WINDOWS - 1) * STRIDE + HORIZON
for code, df in df_dict.items():
    assert len(df) >= need, f"{code}: have {len(df)} rows, need >= {need}"
print(f"All stocks have enough rows (need >= {need}).")